# ShadowKV++ Semantic Fidelity — KV Cache Reuse Method

**Measures whether splicing cached KV from one prompt prefix into another's computation changes the generated output.**

Uses `DynamicCache.crop()` (transformers 5.x) for zero-overhead KV cache splicing.
The first 75% of prompt tokens are shared (identical), the last 25% are shuffled
to simulate a semantic-approximate match.

### Expected result: ROUGE-L = 1.0 between ref and reuse (KV reuse is mathematically faithful)

In [ ]:
# Install deps
!pip install -q torch transformers datasets accelerate huggingface-hub rouge-score numpy pyarrow

from huggingface_hub import login
login(token='YOUR_HF_TOKEN_HERE')

!git clone https://github.com/Kushalk0677/shadowkv.git
%cd shadowkv/experiments

In [ ]:
# Copy the pipeline into the Colab environment
import json, torch, random, warnings, sys
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings('ignore')

# Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
%%writefile run_fidelity_equiv.py
"""Semantic Fidelity — KV Cache Reuse using DynamicCache API."""
import argparse, json, os, sys, time, random, warnings
warnings.filterwarnings('ignore', category=UserWarning, module='transformers')
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS = {
    'tinyllama': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    'qwen25_15b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'gemma2b': 'google/gemma-2b-it',
}

DATASETS = {
    'samsum': ('knkarthick/samsum', 'train', 'dialogue'),
    'alpaca_eval': ('Thanmay/alpaca_eval', 'eval', 'instruction'),
    'banking77': ('mteb/banking77', 'train', 'text'),
}

def load_model(model_name, device):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    ).to(device).eval()
    return model, tokenizer

def generate_standard(model, input_ids, max_new):
    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids, max_new_tokens=max_new,
            do_sample=False, temperature=None, top_p=None,
            pad_token_id=model.config.pad_token_id or 0,
        )
    return out[0][input_ids.shape[1]:]

def generate_with_reuse(model, prompt_ids_orig, prompt_ids_mod, shared, max_new, eos_id):
    shared = min(shared, len(prompt_ids_orig[0]), len(prompt_ids_mod[0]))
    with torch.no_grad():
        out = model(prompt_ids_orig, use_cache=True)
    cache = out.past_key_values
    cache.crop(shared)
    suffix = prompt_ids_mod[:, shared:]
    if suffix.shape[1] > 0:
        with torch.no_grad():
            out = model(suffix, past_key_values=cache, use_cache=True)
        combined_cache = out.past_key_values
        total_pos = shared + suffix.shape[1]
        start_tok = suffix[:, -1:]
    else:
        combined_cache = cache
        total_pos = shared
        start_tok = prompt_ids_orig[:, shared - 1:shared]
    if total_pos < 1:
        return generate_standard(model, prompt_ids_mod, max_new)
    kv = combined_cache
    kv.crop(total_pos - 1)
    inp = start_tok
    gen_ids = []
    for _ in range(max_new):
        with torch.no_grad():
            outs = model(input_ids=inp, past_key_values=kv, use_cache=True)
        nid = outs.logits[0, -1].argmax(-1, keepdim=True).unsqueeze(0)
        inp = nid
        kv = outs.past_key_values
        gen_ids.append(nid.item())
        if nid.item() == eos_id:
            break
    return torch.tensor(gen_ids)

def make_shuffled_ids(ids, ratio=0.75):
    split = int(len(ids) * ratio)
    if len(ids) - split < 4:
        split = len(ids) - 4
    if split < 4:
        return ids, len(ids)
    prefix = ids[:split]
    suffix = ids[split:]
    shuffled = suffix.copy()
    random.shuffle(shuffled)
    return prefix + shuffled, split

def run(args):
    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    all_results = []
    random.seed(42)
    device = args.device
    for mk in args.models:
        model, tok = load_model(MODELS[mk], device)
        eos_id = tok.eos_token_id or 0
        for dk in args.datasets:
            print(f'\n=== {mk} on {dk} ===')
            samples = []
            try:
                from datasets import load_dataset
                info = DATASETS[dk]
                if len(info) == 4:
                    ds = load_dataset(info[0], info[1], split=info[2], trust_remote_code=True)
                else:
                    ds = load_dataset(info[0], split=info[1], trust_remote_code=True)
                for i in range(min(args.n_samples, len(ds))):
                    t = str(ds[i][info[-1]])
                    if len(t) > 20:
                        samples.append(t[:512])
            except Exception as e:
                print(f'  Dataset error: {e}')
                continue
            print(f'  {len(samples)} samples')
            if not samples:
                continue
            for idx, payload in enumerate(samples):
                ids_1d = tok.encode(payload, truncation=True, max_length=384)
                if len(ids_1d) < 16:
                    continue
                ids_mod_1d, shared = make_shuffled_ids(ids_1d)
                ids_orig = torch.tensor([ids_1d], device=device)
                ids_mod = torch.tensor([ids_mod_1d], device=device)
                exact_ids = generate_standard(model, ids_orig, args.max_gen_tokens)
                exact_text = tok.decode(exact_ids, skip_special_tokens=True)
                ref_ids = generate_standard(model, ids_mod, args.max_gen_tokens)
                ref_text = tok.decode(ref_ids, skip_special_tokens=True)
                reuse_ids = generate_with_reuse(model, ids_orig, ids_mod, shared,
                                                 args.max_gen_tokens, eos_id)
                reuse_text = tok.decode(reuse_ids, skip_special_tokens=True)
                all_results.append({
                    'model': mk, 'dataset': dk, 'sample_idx': idx,
                    'shared_tokens': shared,
                    'total_tokens_orig': len(ids_1d),
                    'exact_text': exact_text,
                    'ref_text': ref_text,
                    'reuse_text': reuse_text,
                })
                if (idx + 1) % 10 == 0:
                    print(f'  [{idx+1}/{len(samples)}]')
        path = out_dir / f'{mk}_results.json'
        with open(path, 'w') as f:
            json.dump([r for r in all_results if r['model'] == mk], f, indent=2)
        print(f'Saved -> {path}')
        del model, tok
        if device.startswith('cuda'):
            torch.cuda.empty_cache()
    path = out_dir / 'all_results.json'
    with open(path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'Total: {len(all_results)} -> {path}')

if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--device', default='cuda:0')
    p.add_argument('--models', nargs='+', default=['tinyllama'], choices=MODELS.keys())
    p.add_argument('--datasets', nargs='+', default=['samsum', 'alpaca_eval', 'banking77'])
    p.add_argument('--n_samples', type=int, default=32)
    p.add_argument('--max_gen_tokens', type=int, default=64)
    p.add_argument('--output_dir', default='fidelity_equiv_results')
    args = p.parse_args()
    run(args)

In [ ]:
# Run on T4 — 3 models, 3 datasets, 32 samples each (~15 min on T4)
!python run_fidelity_equiv.py \
  --device cuda:0 \
  --models tinyllama qwen25_15b gemma2b \
  --datasets samsum alpaca_eval banking77 \
  --n_samples 32 \
  --max_gen_tokens 64 \
  --output_dir fidelity_results_gpu

In [ ]:
# Evaluate
from rouge_score import rouge_scorer
import json

r = json.load(open('fidelity_results_gpu/all_results.json'))
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# KV fidelity (ref vs reuse)
matches = sum(1 for x in r if x['ref_text'] == x['reuse_text'])
print(f'KV Fidelity: {matches}/{len(r)} exact matches ({matches/len(r)*100:.1f}%)')

# Prompt sensitivity (exact vs ref)
scores = []
for x in r:
    if x['exact_text'] and x['ref_text']:
        s = scorer.score(x['exact_text'], x['ref_text'])['rougeL'].fmeasure
        scores.append(s)
avg = sum(scores)/len(scores) if scores else 0
print(f'Prompt sensitivity (exact vs ref ROUGE-L): {avg:.4f}')

print(f'\nResults by model:')
by_model = {}
for x in r:
    by_model.setdefault(x['model'], []).append(x)
for mk, items in by_model.items():
    m = sum(1 for x in items if x['ref_text'] == x['reuse_text'])
    avg_s = sum(scorer.score(x['exact_text'], x['ref_text'])['rougeL'].fmeasure 
                for x in items if x['exact_text'] and x['ref_text']) / len(items)
    print(f'  {mk}: {m}/{len(items)} KV matches, ROUGE(exact,ref)={avg_s:.4f}')

In [ ]:
# Download results
from google.colab import files
!zip -r /content/fidelity_results.zip fidelity_results_gpu
files.download('/content/fidelity_results.zip')